# TurtleBot3 AutoRace 2020: Interactive Perception Playground

Welcome to the beginner-friendly computer vision notebook for **AutoRace 2020**.
This notebook allows you to experiment with each step of the perception algorithms without needing to spin up Gazebo, Docker, or ROS 2.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Traffic Light Detection (HSV Color Space Masking)
The camera takes an RGB image of the traffic light at `(1.30, -1.95)`. We crop the upper right ROI and segment the Green circle using HSV thresholding.

In [ ]:
# Create synthetic traffic light image
tl_img = np.zeros((200, 100, 3), dtype=np.uint8)
cv2.circle(tl_img, (50, 40), 20, (0, 0, 80), -1)   # Red off
cv2.circle(tl_img, (50, 100), 20, (0, 80, 80), -1) # Yellow off
cv2.circle(tl_img, (50, 160), 20, (30, 240, 50), -1) # GREEN ON!

# Convert BGR to HSV
hsv = cv2.cvtColor(tl_img, cv2.COLOR_BGR2HSV)
lower_green = np.array([35, 70, 70])
upper_green = np.array([85, 255, 255])
mask_green = cv2.inRange(hsv, lower_green, upper_green)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(cv2.cvtColor(tl_img, cv2.COLOR_BGR2RGB))
ax1.set_title("Traffic Light Input")
ax2.imshow(mask_green, cmap='gray')
ax2.set_title(f"HSV Green Mask (Active Pixels: {cv2.countNonZero(mask_green)})")
plt.show()

## 2. Traffic Sign Matching (SIFT + FLANN + RANSAC Homography)
How does the robot locate a sign and outline it with **only a clean yellow bounding box**?

In [ ]:
tmpl_path = '../turtlebot3_autorace_detect/image/intersection.png'
tmpl = cv2.imread(tmpl_path)
sift = cv2.SIFT_create()
kp_tmpl, des_tmpl = sift.detectAndCompute(tmpl, None)

print(f"Sign Template: {tmpl.shape}")
print(f"Extracted SIFT Keypoints: {len(kp_tmpl)}")

# Simulate camera scene containing scaled sign
scene = np.ones((480, 640, 3), dtype=np.uint8) * 120
scaled_sign = cv2.resize(tmpl, (120, 120))
scene[100:220, 350:470] = scaled_sign

kp_scene, des_scene = sift.detectAndCompute(scene, None)

# FLANN Matching
index_params = dict(algorithm=1, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des_tmpl, des_scene, k=2)
good = [m for m, n in matches if m.distance < 0.78 * n.distance]

# Homography & Bounding Box
src_pts = np.float32([kp_tmpl[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

h, w = tmpl.shape[:2]
corners = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)
dst_corners = cv2.perspectiveTransform(corners, M)

output = scene.copy()
# Draw ONLY yellow boundary box and label
cv2.polylines(output, [np.int32(dst_corners)], isClosed=True, color=(0, 255, 255), thickness=3)
cv2.putText(output, "INTERSECTION", (350, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.title(f"Clean Yellow Bounding Box Output (Matches: {len(good)})")
plt.show()

## 3. Pure Pursuit Steering Math Formula
Given current robot pose $(x, y, \theta)$ and target waypoint $(g_x, g_y)$:
1. Heading angle to waypoint: $\alpha = \text{atan2}(g_y - y, g_x - x) - \theta$
2. Lookahead distance: $L_d = \sqrt{(g_x - x)^2 + (g_y - y)^2}$
3. Curvature: $\kappa = \frac{2 \sin(\alpha)}{L_d}$
4. Angular speed: $\omega = v \cdot \kappa$

In [ ]:
import math

def pure_pursuit(robot_x, robot_y, robot_yaw, target_x, target_y, linear_v):
    dx = target_x - robot_x
    dy = target_y - robot_y
    lookahead = math.hypot(dx, dy)
    alpha = math.atan2(dy, dx) - robot_yaw
    # Wrap angle between -pi and +pi
    alpha = math.atan2(math.sin(alpha), math.cos(alpha))
    curvature = (2.0 * math.sin(alpha)) / max(0.05, lookahead)
    angular_w = linear_v * curvature
    return lookahead, alpha, angular_w

Ld, alpha, w = pure_pursuit(0.8, -1.75, 0.0, 1.30, -1.75, 0.10)
print(f"Lookahead Distance: {Ld:.2f} m")
print(f"Heading Error (Alpha): {math.degrees(alpha):.1f} deg")
print(f"Commanded Angular Velocity: {w:.3f} rad/s")